# Cox Proportional Hazards 

El **Cox Proportional Hazards (Cox PH)** es un modelo semi-paramétrico de
análisis de supervivencia que estima el riesgo de fallo en función de
covariables sin asumir una forma paramétrica para la baseline hazard h₀(t):

    h(t|X) = h₀(t) · exp(β'X)

A diferencia de los modelos de regresión del proyecto (NB, SVR, DT, RF, XGB),
Cox PH no predice el RUL directamente — estima la función de supervivencia
S(t|X) = P(T > t|X) y deriva el RUL mediante la curva de muerte
F(t|X) = 1 - S(t|X): el primer tiempo t* donde F(t*) = confidence_threshold
define el ciclo de fallo predicho, y RUL = t* - t_stop_actual.

**¿Por qué Cox PH para RUL?**
Cox PH es metodológicamente correcto para datos de degradación con evento
terminal observado — cada motor falla exactamente una vez, lo que se alinea
naturalmente con el marco de supervivencia. Es el modelo de referencia en la
literatura de PHM basada en supervivencia y permite comparar directamente con
los modelos de regresión del proyecto.

**Abordaje en este proyecto**
Se usa `survival::coxph` con estimación de baseline por el método de Breslow
(no paramétrico) o splines cúbicos. La función de supervivencia S(t|X) se
computa via `predict_survival_function()` de lifelines. t_stop de cada ventana
se usa como duración — tiempo absoluto acumulado desde el arranque del motor.

**Hiperparámetros considerados**

| Hiperparámetro | Rango explorado | Justificación |
|----------------|----------------|---------------|
| `feature_set` | A, B, C, D | Evalúa si features adicionales mejoran la estimación de β |
| `window_size` | 20, 25, 30 | Horizonte temporal de cada ventana |
| `n_components` | 10, 15, 20 | Dimensionalidad del espacio de covariables |
| `clipping_threshold` | 115, 120, 125 | Techo del RUL predicho |
| `baseline_estimation_method` | breslow, spline | Estimación no paramétrica vs paramétrica de h₀(t) |
| `n_baseline_knots` | 3, 5 | Knots para baseline spline — irrelevante con breslow |
| `penalizer` | 0.0, 0.1, 1.0 | Regularización L2 sobre β |
| `l1_ratio` | 0.0, 0.5, 1.0 | Mixing L1/L2 — solo activo con penalizer > 0 |
| `confidence_threshold` | 0.3, 0.5, 0.95 | Umbral de F(t) para declarar fallo predicho |

##  Resultados GGS y diagnóstico

**Resumen del GGS**
- Configuraciones evaluadas: 11,664
- Exitosas: 2,592 (22.2%) — Fallidas: 9,072 (77.8%)
- Folds: 5 (GroupKFold por motor)

**Top 10 configuraciones**

| feature_set | window_size | n_components | clipping_threshold | baseline | penalizer | conf_thresh | S-Score | MAE | RMSE | C-Index |
|-------------|-------------|--------------|-------------------|----------|-----------|-------------|---------|-----|------|---------|
| A | 20 | 20 | 125 | breslow | 0.1 | 0.3 | 7,182 | 32.5 | 49.4 | 0.543 |
| A | 20 | 20 | 125 | breslow | 0.1 | 0.3 | 7,182 | 32.5 | 49.4 | 0.543 |
| A | 20 | 20 | 120 | breslow | 0.1 | 0.3 | 7,182 | 34.5 | 49.5 | 0.545 |
| A | 20 | 20 | 120 | breslow | 0.1 | 0.3 | 7,182 | 34.5 | 49.5 | 0.545 |
| A | 20 | 20 | 115 | breslow | 0.1 | 0.3 | 7,182 | 36.7 | 49.8 | 0.546 |
| A | 20 | 20 | 115 | breslow | 0.1 | 0.3 | 7,182 | 36.7 | 49.8 | 0.546 |
| A | 20 | 10 | 125 | breslow | 0.1 | 0.3 | 7,193 | 32.5 | 49.4 | 0.543 |
| A | 20 | 10 | 125 | breslow | 0.1 | 0.3 | 7,193 | 32.5 | 49.4 | 0.543 |
| A | 20 | 15 | 125 | breslow | 0.1 | 0.3 | 7,193 | 32.5 | 49.4 | 0.543 |
| A | 20 | 15 | 125 | breslow | 0.1 | 0.3 | 7,193 | 32.5 | 49.4 | 0.543 |

### Diagnóstico: incompatibilidad estructural

**El problema de la tasa de eventos**

El pipeline de ventanas deslizantes con paso 1 transforma el dataset así:

```
140 motores × ~170 ciclos promedio = ~23,800 ventanas
Eventos observados: 140 (uno por motor, en el último ciclo)
Tasa de eventos: 140 / 23,800 ≈ 0.59%
```

Cox PH ve 23,800 observaciones de las cuales 23,660 son censuras. La partial
likelihood de Cox no tiene suficiente información para estimar β — la superficie
de likelihood es casi plana y el optimizador converge hacia β ≈ 0 para todas
las componentes PCA.

**Consecuencia en la baseline hazard de Breslow**

Con β ≈ 0, todas las ventanas producen la misma curva de supervivencia:

```
S(t|X) ≈ S₀(t)  para todas las ventanas
```

La baseline de Breslow coloca masa únicamente en los tiempos donde ocurren
eventos — concentrados en los últimos ciclos de cada motor. El resultado es
S(t) = 1.0 durante el 99.4% de la vida del motor, con caída abrupta solo
al final. F(t) raramente supera 0.3 — explicando por qué `confidence_threshold
= 0.3` es el único umbral que produce predicciones no-NaN, y por qué el
77.8% de configuraciones falla completamente.

**Por qué spline falla más que breslow**

La baseline spline requiere estimar adicionalmente los parámetros de los knots.
Con 0.4% evento rate, no hay suficiente información para estimar simultáneamente
β y la forma de la baseline — el optimizador falla en la mayoría de casos.
Breslow produce siempre una estimación válida aunque sea casi plana.

**Este problema no depende de la representación de features**

Un resultado crítico: la incompatibilidad no es una limitación del pipeline
de ventanas deslizantes ni de las features PCA — es una propiedad fundamental
del dataset. Con sensores crudos el problema es exactamente el mismo:

```
140 motores × ~170 ciclos = ~23,800 filas
Tasa de eventos: 140 / 23,800 ≈ 0.59% — idéntica
```

La tasa de eventos depende exclusivamente de la naturaleza del dataset —
un evento por motor por definición en C-MAPSS FD001 — no de cómo se
construyen las features. Esto revela una tensión fundamental e irresoluble:

> *Para explotar la información temporal de los sensores se necesita el
> formato fila-por-ciclo. Pero ese formato produce una tasa de eventos
> estructuralmente baja que hace que Cox PH sea inviable. El formato que
> enriquece la información destruye la señal de supervivencia.*

La única alternativa sería usar una fila por motor — el formato clásico de
supervivencia — pero eso elimina toda la riqueza temporal de los sensores,
que es precisamente lo que hace útil el pipeline de degradación.

**El C-Index como confirmación**

C-Index ≈ 0.544 — prácticamente aleatorio (0.5 = random). El modelo no
discrimina entre motores en distintos estados de degradación. Con β ≈ 0,
todas las predicciones son casi idénticas independientemente del estado real
del motor.

### Comparación acumulada

| Métrica | NB | DT | RF | SVR | XGB | **CoxPH** |
|---------|----|----|-----|-----|-----|-----------|
| S-Score | 2.447 | 2.755 | 1.883 | 1.825 | 1.782 | **7,182** |
| MAE | 10.39 | 7.79 | 6.98 | 6.75 | 7.10 | **32.5** |
| RMSE | 13.36 | 12.22 | 10.62 | 10.24 | 10.45 | **49.4** |
| C-Index | 0.908 | 0.893 | 0.909 | 0.915 | 0.912 | **0.544** |

### Conclusión

Cox PH no es competitivo para este dataset y pipeline. La tasa de eventos
de 0.59% hace que el modelo sea estadísticamente inviable — no por limitaciones
de implementación sino por incompatibilidad estructural entre el formato
fila-por-ciclo requerido para explotar los sensores y el formato fila-por-motor
que necesita Cox para estimar β de forma significativa.

**No se seleccionan hiperparámetros de producción.** Cox PH queda documentado
como resultado negativo con valor metodológico: confirma que los modelos de
supervivencia semi-paramétricos clásicos requieren tasas de eventos
significativamente mayores que las producidas por pipelines de ventanas
deslizantes sobre C-MAPSS FD001.

In [4]:
import pandas as pd

results_path = 'outputs/ggs/results/CoxPHModel_59d4a8b5_20260512_1106.csv'

cols_params = ['feature_set', 'window_size', 'n_components', 'clipping_threshold', 
               'baseline_estimation_method', 'n_baseline_knots', 'penalizer', 
               'l1_ratio', 'confidence_threshold', 'mean_S_score', 'mean_C_index', 
               'mean_MAE', 'mean_RMSE']


df_results = pd.read_csv(results_path)

total = len(df_results)
exitosas = df_results['mean_S_score'].notna().sum()
fallidas = df_results['mean_S_score'].isna().sum()
n_duplicados = df_results.duplicated(subset=cols_params).sum()

print(f"Total configuraciones: {total}")
print(f"Exitosas:              {exitosas}")
print(f"Fallidas (NaN):        {fallidas}")
print(f"Duplicados:            {n_duplicados}")
print(f"Únicas:                {total - n_duplicados}")

Total configuraciones: 11664
Exitosas:              2592
Fallidas (NaN):        9072
Duplicados:            0
Únicas:                11664


In [3]:
df_top = (
    df_results
    .dropna(subset=['mean_S_score'])
    .sort_values('mean_S_score', ascending=True)
    .head(10)
    .reset_index(drop=True)
)

df_top[cols_params]

,feature_set,window_size,n_components,clipping_threshold,baseline_estimation_method,n_baseline_knots,penalizer,l1_ratio,confidence_threshold,mean_S_score,mean_C_index,mean_MAE,mean_RMSE
0,A,20,20,125,breslow,3,0.1,0.0,0.3,7181.622052,0.543478,32.529942,49.373263
1,A,20,20,125,breslow,5,0.1,0.0,0.3,7181.622052,0.543478,32.529942,49.373263
2,A,20,20,120,breslow,3,0.1,0.0,0.3,7181.842354,0.544783,34.543207,49.456894
3,A,20,20,120,breslow,5,0.1,0.0,0.3,7181.842354,0.544783,34.543207,49.456894
4,A,20,20,115,breslow,5,0.1,0.0,0.3,7182.274925,0.546284,36.714807,49.771556
5,A,20,20,115,breslow,3,0.1,0.0,0.3,7182.274925,0.546284,36.714807,49.771556
6,A,20,10,125,breslow,3,0.1,0.0,0.3,7192.910285,0.543419,32.531975,49.377522
7,A,20,10,125,breslow,5,0.1,0.0,0.3,7192.910285,0.543419,32.531975,49.377522
8,A,20,15,125,breslow,5,0.1,0.0,0.3,7192.939122,0.543419,32.532400,49.378036
9,A,20,15,125,breslow,3,0.1,0.0,0.3,7192.939122,0.543419,32.532400,49.378036
